# 06 — ICA Artifact Rejection

Four-step pipeline:
1. **Fit** ICA on the designated window (`use_for_ica: true` in config)
2. **Label** components automatically with ICLabel
3. **Inspect** interactively — confirm or adjust auto-selections
4. **Apply** — project out excluded components from all epoching windows

**Requires:** Qt5 backend for interactive plots

**Input:** `<subject>_<ica_window>-epo.fif`  
**Output:** `<subject>_ica.fif`, `<subject>_iclabel.json`, `<subject>_<window>_clean-epo.fif`

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib
matplotlib.use('Qt5Agg')

from eeg_toolkit import (
    load_config,
    find_subjects,
    fit_ica_subject,
    label_ica_subject,
    inspect_ica_subject,
    apply_ica_subject,
)

# ── Update this path to point to your experiment config ──
cfg = load_config('../configs/your_experiment.yaml')

subjects = find_subjects(cfg)
print(f"Subjects: {len(subjects)}")

In [ ]:
# ── Step 1: Fit ICA (test subject) ──
test_subject = subjects[0]
print(f"\nFitting ICA for {test_subject}\n")

fit_ica_subject(cfg, test_subject, overwrite=False, verbose=True)

In [ ]:
# ── Step 2: Run ICLabel (test subject) ──
print(f"Running ICLabel for {test_subject}\n")

label_ica_subject(cfg, test_subject, overwrite=True, verbose=True)

In [ ]:
# ── Step 3: Interactive inspection (test subject) ──
# Red titles = auto-excluded by ICLabel.
# Click topographies to toggle exclusion.
# Close BOTH windows when done.
inspect_ica_subject(cfg, test_subject, verbose=True)

In [ ]:
# ── Run steps 1–2 for all subjects ──
from eeg_toolkit import fit_ica_all, label_ica_all

print("--- Fitting ICA for all subjects ---")
fit_summary = fit_ica_all(cfg, overwrite=False, verbose=True)

print("\n--- Running ICLabel for all subjects ---")
label_summary = label_ica_all(cfg, overwrite=False, verbose=True)

In [ ]:
# ── Step 3: Interactive inspection for all subjects ──
from eeg_toolkit import inspect_ica_all

inspect_summary = inspect_ica_all(cfg, verbose=True)

In [ ]:
# ── Step 4: Apply ICA to all epoching windows ──
from eeg_toolkit import apply_ica_all

apply_summary = apply_ica_all(cfg, overwrite=False, verbose=True)

In [ ]:
# ── Verification: check all clean epoch files exist ──
import json
from eeg_toolkit import find_subjects, get_clean_epochs_path
from eeg_toolkit.ica import _get_iclabel_path

subjects = find_subjects(cfg)
windows  = [w.name for w in cfg.epoching_windows]

print(f"=== ICA Pipeline Summary ({len(subjects)} subjects) ===\n")
all_ok = True
for subj in subjects:
    paths_ok = all(get_clean_epochs_path(cfg, subj, w).exists() for w in windows)
    n_excl = "?"
    iclabel_path = _get_iclabel_path(cfg, subj)
    if iclabel_path.exists():
        with open(iclabel_path) as f:
            log = json.load(f)
        n_excl = len([c for c in log['components'] if c['auto_excluded']])
    status = "OK" if paths_ok else "INCOMPLETE"
    if status != "OK":
        all_ok = False
    window_status = ", ".join(f"{w}={'OK' if get_clean_epochs_path(cfg, subj, w).exists() else 'MISSING'}" for w in windows)
    print(f"  [{status:10s}] {subj}: ICLabel excluded={n_excl}, {window_status}")

print(f"\n{'All subjects complete!' if all_ok else 'Some subjects incomplete — check above.'}")